In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import(
    current_timestamp, lit, sum, count, round as spark_round
)
import builtins

spark = SparkSession.builder.appName("GoldAggregation").getOrCreate()

#Step 1: Read from Silver table 
print('Reading from Silver layer...')
silver_df = spark.read.format("delta").table("silver_payments")
print(f"Silver row count: {silver_df.count()}")

#Step 2: Total revenue and transaction count by currency
print("\nKPI 1 - Revenue by currency:")
revenue_by_currency = silver_df \
    .groupBy("currency") \
    .agg(   
         spark_round(sum("amount"), 2).alias("total_revenue"),
         count("payment_id").alias("transaction_count")
    )
revenue_by_currency.show()

#Step 3: Transaction count by status
print("KPI 2 - Transactions by status:")
by_status = silver_df \
    .groupby("status") \
    .agg(
        count("payment_id").alias("transaction_count"),
        spark_round(sum("amount"), 2).alias("total_amount")
    )
by_status.show()

#Step 4: Revenue by payment method
print("KPI 3 - Revenue by payment method:")
by_method = silver_df \
    .groupBy("payment_method") \
    .agg(
        count("payment_id").alias("transaction_count"),
        spark_round(sum("amount"), 2).alias("total_revenue"),
        
    )
by_method.show()

#Step 5: Fraud rate
#Fraud rate = failed transactions / total transactions
print("KPI 4 - Fraud and failure rate:")
total = silver_df.count()
failed = silver_df.filter("status='failed'").count()
fraud_rate = builtins.round((failed/total)*100, 2)
print(f"Total transactions: {total}")
print(f"Failed transactions: {failed}")
print(f"Fraud rate: {fraud_rate}%")

#Step 6: Build Gold summary table
gold_df = silver_df \
    .groupBy("currency", "payment_method", "status") \
    .agg(
        count("payment_id").alias("transaction_count"),
        spark_round(sum("amount"), 2).alias("total_revenue")
    ) \
    .withColumn("processed_timestamp", current_timestamp()) \
    .withColumn("layer", lit("gold"))

print("\nGold layer - aggregated KPIs:")
gold_df.show(truncate=False)

#Step 7: Save to Delta Lake as Gold table
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_payments")

print('Gold table saved successfully!')
print(f"Total KPI rows:{gold_df.count()}")

Reading from Silver layer...
Silver row count: 5

KPI 1 - Revenue by currency:
+--------+-------------+-----------------+
|currency|total_revenue|transaction_count|
+--------+-------------+-----------------+
|     USD|       407.49|                3|
|     EUR|        567.0|                1|
|     GBP|        89.99|                1|
+--------+-------------+-----------------+

KPI 2 - Transactions by status:
+---------+-----------------+------------+
|   status|transaction_count|total_amount|
+---------+-----------------+------------+
|succeeded|                3|      950.99|
|   failed|                1|        23.5|
|  pending|                1|       89.99|
+---------+-----------------+------------+

KPI 3 - Revenue by payment method:
+--------------+-----------------+-------------+
|payment_method|transaction_count|total_revenue|
+--------------+-----------------+-------------+
|   credit_card|                2|       383.99|
|    debit_card|                1|         23.5|
| ban